# NLPG: Natural Language Processing
## Practical 4 — N-gram estimation, perplexity and source identification

**University of York**

### Central question

> **What can word n-gram language models tell us about where a passage came from?**

Week 1 used finite-history models mainly to generate samples and to compare character- and word-level representations. This practical does **not** repeat those experiments. Here the emphasis is on **estimating probabilities from finite evidence, evaluating held-out text, and using perplexity to make a prediction**.

There is nothing to submit. Take notes separately on your predictions, observations and explanations. Keep them and bring them to this week's summing-up lecture (L2), where we may compare and discuss your findings.

Only **Part A — authorship from raw word n-grams** is required during the scheduled practical. The topic and stopword investigations are extensions or work to continue afterwards.


## The mini-corpus

The supplied corpus crosses three authors with three situational topics:

| | Travel | Eating | Person description |
|---|---|---|---|
| Jane Austen | 4 passages | 4 passages | 4 passages |
| Arthur Conan Doyle | 4 passages | 4 passages | 4 passages |
| Robert Louis Stevenson | 4 passages | 4 passages | 4 passages |

Within each cell, three passages are training data and one is test data. The labels remain visible. **Held out** means excluded from fitting; it does not mean that the label must be concealed from you.

The sources are *Pride and Prejudice*, *The Adventures of Sherlock Holmes*, and *Treasure Island*. Consequently, a result labelled “authorship” is evidence about these particular sources. It does not prove that the method has isolated an author's style across all their writing.


In [ ]:
from collections import Counter
from pathlib import Path
import json
import math
import re

import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.lm import MLE, Lidstone, Vocabulary
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_both_ends
from nltk.util import ngrams

DATA_FILE = Path("NLPG_Week4_mini_corpus.json")

if not DATA_FILE.exists():
    try:
        from google.colab import files
        print(f"Please upload {DATA_FILE.name} from the VLE.")
        files.upload()
    except ImportError as exc:
        raise FileNotFoundError(
            f"Put {DATA_FILE.name} in the same folder as this notebook."
        ) from exc

records = json.loads(DATA_FILE.read_text(encoding="utf-8"))
corpus = pd.DataFrame(records)

print(f"Loaded {len(corpus)} passages.")
display(corpus[["snippet_id", "author_key", "topic", "split", "word_count"]].head())


In [ ]:
# Check the 3 × 3 design and the train/test split.
display(pd.crosstab(
    [corpus["author_key"], corpus["topic"]],
    corpus["split"],
))

display(
    corpus.groupby(["author_key", "topic", "split"])["word_count"]
          .agg(["count", "min", "max"])
)


## Word tokenisation

We build **word** n-grams. The code lower-cases the prose and extracts sequences of letters, retaining internal apostrophes. Punctuation is therefore not a token in these models, but common function words are retained unless an extension explicitly removes them.

For simplicity, each excerpt is treated as one word sequence. Once punctuation is removed, an n-gram can therefore cross an original sentence boundary. This is another modelling decision rather than a neutral property of the text.

Before running the next cell, predict what information word bigrams and trigrams might preserve that isolated words do not.


In [ ]:
WORD_PATTERN = re.compile(r"[^\W\d_]+(?:['’][^\W\d_]+)*", flags=re.UNICODE)

def word_tokens(text, remove_stopwords=False):
    # Return lower-case word tokens; punctuation is not a word token.
    tokens = WORD_PATTERN.findall(text.lower())
    if remove_stopwords:
        tokens = [token for token in tokens if token not in ENGLISH_STOP_WORDS]
    return tokens


example = corpus.iloc[0]["text"]
print(word_tokens(example)[:35])


## Perplexity: how surprised is the model?

For a held-out token sequence containing \(K\) predictions, its average negative log probability is

$$
H(T,M)=-\frac{1}{K}\sum_{i=1}^{K}\log_2 P_M(w_i\mid h_i),
$$

where \(h_i\) is the retained history for the prediction of \(w_i\). Perplexity is

$$
PP(T\mid M)=2^{H(T,M)}.
$$

A **lower** perplexity means that the model was less surprised by the held-out sequence.

In this practical, perplexity is used **ordinally**. For the **same passage**, scored by candidate models built with the same order, tokenisation, smoothing and vocabulary policy, we choose the model with the lowest perplexity. Perplexity is not a metric distance: a passage is not meaningfully “20 perplexity units closer” to one author.

Do not casually compare absolute perplexity values produced with different tokenisations or vocabularies. When comparing bigrams and trigrams below, compare their **source-identification results** rather than treating the two perplexity scales as interchangeable.


## Why zero probabilities force us to smooth

Maximum-likelihood estimation assigns probability 0 to an n-gram that was never observed in training. If a held-out passage contains even one such event, the probability of the complete passage becomes 0 and its perplexity becomes infinite.

That is a very strong conclusion. Usually we mean **“not observed in this finite training sample”**, not **“linguistically impossible”**.

We will use one modest remedy: **Lidstone smoothing** with a small added count, \(\gamma\) = 0.1. Conceptually, smoothing reserves or reallocates some probability mass for unseen events. It changes the estimate; it does not make sparse evidence disappear.




For an \(n\)-gram model, Lidstone smoothing estimates

$$
P_{\text{Lidstone}}(w_i\mid h_i)
=
\frac{C(h_i,w_i)+\gamma}
     {C(h_i)+\gamma |V|}
$$

where:

- $h_i$ is the preceding \(n-1\)-word context;
- C($h_i,w_i$) is the count of the complete \(n\)-gram;
- $C(h_i)=\sum_{w\in V}C(h_i,w)$ is the total count of that context;
- $|V|$ is the prediction vocabulary size;
- $\gamma>0$ is the added pseudo-count.

Thus an unseen continuation receives

$$
P_{\text{Lidstone}}(w_i\mid h_i)
=
\frac{\gamma}{C(h_i)+\gamma|V|}.
$$

When $\gamma=1$, Lidstone smoothing becomes **Laplace (add-one) smoothing**. In our notebook, $\gamma=0.1$.

We also need a policy for unseen words. For each experiment we:

1. construct one vocabulary from the pooled **training data only**;
2. replace words seen fewer than twice with `<UNK>`;
3. map unseen test words to the same `<UNK>` token;
4. use that shared vocabulary for every candidate model.

The shared vocabulary matters because candidate models with different vocabularies would not provide a fair basis for the ordinal comparison.

In [ ]:
UNK = "<UNK>"
UNK_CUTOFF = 2
GAMMA = 0.1

def make_vocabulary_policy(sequences, cutoff=UNK_CUTOFF):
    # Choose known words from pooled training sequences.
    counts = Counter(token for sequence in sequences for token in sequence)
    known_words = {token for token, count in counts.items() if count >= cutoff}
    return known_words


def apply_vocabulary(tokens, known_words):
    # Map rare or unseen words to a shared unknown-word token.
    return [token if token in known_words else UNK for token in tokens]


def fit_language_model(sequences, n, known_words, gamma=None):
    # Fit an exact-order MLE or Lidstone-smoothed NLTK language model.
    mapped = [apply_vocabulary(sequence, known_words) for sequence in sequences]
    training_ngrams, padded_vocabulary = padded_everygram_pipeline(n, mapped)

    # A fresh Vocabulary object gives every candidate model the same members.
    vocabulary = Vocabulary(sorted(known_words) + [UNK], unk_cutoff=1)
    model = MLE(n, vocabulary=vocabulary) if gamma is None \
        else Lidstone(gamma, n, vocabulary=vocabulary)
    model.fit(training_ngrams, padded_vocabulary)
    return model


def exact_order_ngrams(tokens, n):
    # Create one padded n-gram event per prediction at order n.
    return list(ngrams(pad_both_ends(tokens, n=n), n))


def score_perplexity(model, tokens, n, known_words):
    mapped = apply_vocabulary(tokens, known_words)
    return model.perplexity(exact_order_ngrams(mapped, n))


### A small demonstration before the main experiment

The tiny training sample below has seen `the cat sat`, but not `the cat slept`. Try to predict what the unsmoothed and smoothed bigram models will do before running the cell.


In [ ]:
toy_training = [
    ["the", "cat", "sat"],
    ["the", "dog", "ran"],
]
toy_test = ["the", "cat", "slept"]

toy_known = make_vocabulary_policy(toy_training, cutoff=1)
toy_mle = fit_language_model(toy_training, n=2, known_words=toy_known, gamma=None)
toy_smoothed = fit_language_model(toy_training, n=2, known_words=toy_known, gamma=GAMMA)

print("Held-out sequence:", " ".join(toy_test))
print("Unsmoothed MLE perplexity:", score_perplexity(toy_mle, toy_test, 2, toy_known))
print("Lidstone-smoothed perplexity:", round(score_perplexity(toy_smoothed, toy_test, 2, toy_known), 2))


The finite smoothed value is not evidence that the held-out sentence is good English. It only shows that smoothing no longer turns an unseen event into an absolute claim of impossibility.

For the source-identification experiment we use the same $\gamma$ for every candidate model. We are not trying to establish that 0.1 is universally optimal.


In [ ]:
def train_class_models(training_rows, label_column, n,
                       remove_stopwords=False, gamma=GAMMA):
    # Train one comparable smoothed word n-gram model per label.
    tokenised = {
        row.snippet_id: word_tokens(row.text, remove_stopwords)
        for row in training_rows.itertuples()
    }

    # The vocabulary policy is determined once from all training passages.
    known_words = make_vocabulary_policy(tokenised.values())

    models = {}
    for label, group in training_rows.groupby(label_column):
        sequences = [tokenised[snippet_id] for snippet_id in group["snippet_id"]]
        models[label] = fit_language_model(
            sequences, n=n, known_words=known_words, gamma=gamma
        )
    return models, known_words


def evaluate_by_perplexity(data, label_column, n,
                           remove_stopwords=False, gamma=GAMMA):
    # Predict each held-out label using the lowest candidate perplexity.
    training = data[data["split"] == "train"].copy()
    testing = data[data["split"] == "test"].copy()

    models, known_words = train_class_models(
        training,
        label_column=label_column,
        n=n,
        remove_stopwords=remove_stopwords,
        gamma=gamma,
    )

    labels = sorted(models)
    rows = []
    for passage in testing.itertuples():
        tokens = word_tokens(passage.text, remove_stopwords)
        scores = {
            label: score_perplexity(models[label], tokens, n, known_words)
            for label in labels
        }
        predicted = min(scores, key=scores.get)
        true_label = getattr(passage, label_column)

        row = {
            "snippet_id": passage.snippet_id,
            "topic": passage.topic,
            "true_label": true_label,
            "predicted_label": predicted,
            "correct": predicted == true_label,
        }
        row.update({f"ppl_{label}": value for label, value in scores.items()})
        rows.append(row)

    return pd.DataFrame(rows)


def summarise(results):
    score_columns = [column for column in results if column.startswith("ppl_")]
    shown = results.copy()
    shown[score_columns] = shown[score_columns].round(1)
    display(shown)
    correct = int(results["correct"].sum())
    print(f"Accuracy: {results['correct'].mean():.1%} ({correct}/{len(results)})")


# Part A — Required: identify authors using raw word n-grams

We train one language model for each author. For the **same held-out passage**, we calculate perplexity under all three candidate models and predict the author whose model gives the **lowest** perplexity:

$$
\widehat{author}(T)=\operatorname*{arg\,min}_{a} PP(T\mid M_a).
$$

Remember: the labels remain visible to you, but test passages were excluded from model fitting.


## A1. Start with bigrams

Before running the cell, predict:

- whether the correct author will usually have the lowest perplexity;
- what textual evidence could make that happen;
- which passages might be difficult.

Run the cell and inspect every row. Do not look only at the final accuracy.


In [ ]:
author_bigrams = evaluate_by_perplexity(
    corpus,
    label_column="author_key",
    n=2,
    remove_stopwords=False,
)
summarise(author_bigrams)


## A2. Compare trigrams

Trigrams retain more local context, but they also divide finite evidence among more specific histories.

Predict first: will trigrams necessarily improve authorship identification on this small corpus? Then run the cell and compare the predictions and the pattern of perplexities.


In [ ]:
author_trigrams = evaluate_by_perplexity(
    corpus,
    label_column="author_key",
    n=3,
    remove_stopwords=False,
)
summarise(author_trigrams)

author_comparison = pd.DataFrame({
    "representation": ["raw word bigrams", "raw word trigrams"],
    "accuracy": [
        author_bigrams["correct"].mean(),
        author_trigrams["correct"].mean(),
    ],
})
display(author_comparison.style.format({"accuracy": "{:.1%}"}))


## A3. Observe and explain

In your notes, consider:

1. Were correct decisions usually decisive, or were some candidate perplexities close?
2. Did the longer context help, hurt, or make no difference to accuracy?
3. Could the models be using character names, recurring subject matter or book-specific vocabulary rather than authorial style?
4. Why would testing on a different work by the same author support a stronger authorship claim?

Completing Part A is sufficient for the scheduled practical.


# From n-grams to features

You have used recurring word sequences as observable textual evidence for distinguishing sources. We can describe such observable properties as **features**.

Here, language models turned n-gram evidence into probabilities and perplexities. In later work, n-gram counts can also be supplied directly as features to classifiers, regressors and other predictive models.

> **A feature is not useful in the abstract. Its usefulness depends on the prediction problem.**

The extensions ask whether the same representations that distinguish authors also help distinguish topics.


# Part B — Extension: identify topics using raw word n-grams

Pool the training passages by topic rather than author. Train `travel`, `eating` and `person_description` models, then choose the lowest-perplexity topic for every held-out passage.

Before running the cell, predict whether topic identification will be easier or harder than authorship identification, and whether bigrams or trigrams will help.


In [ ]:
topic_bigrams_raw = evaluate_by_perplexity(
    corpus, "topic", n=2, remove_stopwords=False
)
topic_trigrams_raw = evaluate_by_perplexity(
    corpus, "topic", n=3, remove_stopwords=False
)

print("Raw word bigrams")
summarise(topic_bigrams_raw)
print("Raw word trigrams")
summarise(topic_trigrams_raw)


# Part C — Extension: remove stopwords and test a hypothesis

**Hypothesis:** removing common function words may make content/topic evidence more prominent.

This is not a guaranteed result. Predict, run, observe and explain.

The transformation also changes adjacency. If

> `the ship sailed into the harbour after the storm`

becomes

> `ship sailed harbour storm`

then `sailed harbour` is a bigram of the **transformed sequence**, even though those words were not adjacent in the original prose.

Before running the cell, predict what this may do to bigram and trigram estimates.


In [ ]:
topic_bigrams_filtered = evaluate_by_perplexity(
    corpus, "topic", n=2, remove_stopwords=True
)
topic_trigrams_filtered = evaluate_by_perplexity(
    corpus, "topic", n=3, remove_stopwords=True
)

print("Stopword-filtered word bigrams")
summarise(topic_bigrams_filtered)
print("Stopword-filtered word trigrams")
summarise(topic_trigrams_filtered)

topic_comparison = pd.DataFrame({
    "representation": [
        "raw bigrams",
        "raw trigrams",
        "stopword-filtered bigrams",
        "stopword-filtered trigrams",
    ],
    "accuracy": [
        topic_bigrams_raw["correct"].mean(),
        topic_trigrams_raw["correct"].mean(),
        topic_bigrams_filtered["correct"].mean(),
        topic_trigrams_filtered["correct"].mean(),
    ],
})
display(topic_comparison.style.format({"accuracy": "{:.1%}"}))


# Part D — Optional: what does stopword removal do to authorship?

Repeat one authorship condition after stopword removal. Predict first. If the result changes, what evidence did the transformation remove or create?


In [ ]:
author_bigrams_filtered = evaluate_by_perplexity(
    corpus, "author_key", n=2, remove_stopwords=True
)
summarise(author_bigrams_filtered)


# Final reflection

Use your notes to answer:

- Which raw word n-gram representation was useful for identifying authors in this mini-corpus?
- Did the same representation work best for topic?
- Did stopword removal support the topic hypothesis? If not, why might it have failed?
- What information did each preprocessing choice preserve, discard or create?
- What stronger dataset would be needed to separate authorial style from one particular book or collection?

The main lesson is broader than this dataset:

> **Preprocessing and representation determine which evidence a model is allowed to use.**

This is one bridge from n-gram modelling to later work on features, classification, semantics and learned representations.
